In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')
if not os.path.exists('/content/drive/MyDrive/Recipes-Recommendation-System'):
    os.makedirs("/content/drive/MyDrive/Recipes-Recommendation-System")
    %cd /content/drive/MyDrive/Recipes-Recommendation-System
    !git clone https://github.com/soheil-housni/Recipes-Recommendation-System.git \
        /content/drive/MyDrive/Recipes-Recommendation-System

else:
    %cd /content/drive/MyDrive/Recipes-Recommendation-System
    !git pull origin main

print(os.getcwd())

Mounted at /content/drive
/content/drive/MyDrive/Recipes-Recommendation-System
remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 7 (delta 4), reused 7 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 2.47 KiB | 0 bytes/s, done.
From https://github.com/soheil-housni/Recipes-Recommendation-System
 * branch            main       -> FETCH_HEAD
   c839d0a..00366a5  main       -> origin/main
Updating c839d0a..00366a5
Fast-forward
 main.ipynb                   | 232 +++++++++++++++++++------------------------
 modules/__init__.py          |   3 +-
 modules/load_parquet_file.py |   2 +-
 scaler.pkl                   | Bin 0 -> 1159 bytes
 4 files changed, 103 insertions(+), 134 deletions(-)
 create mode 100644 scaler.pkl
/content/drive/MyDrive/Recipes-Recommendation-System


In [2]:
!pip install mlflow
!pip install loguru
!pip install optuna

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 72.4 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 97.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.5/838.5 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
import pandas as pd
import numpy as np
import ast
import numpy as np
import torch
import mmh3
from transformers import AutoTokenizer,DistilBertModel
import itertools
import pyarrow.parquet as pq
from torch.utils.data import DataLoader
from sklearn.preprocessing import StandardScaler
import mlflow
from modules import BeforeSplitPreprocessingRecipesFeatures,BeforeSplitPreprocessingUsersFeatures, AfterSplitPreprocessingRecipesFeatures,AfterSplitPreprocessingUsersFeatures, CreationDataset,CollateFunction,set_seed,seed_worker,EncodedHashedEmbeddings,Scaler,split_df,RecommendationModel,Train,BERTCreationDataset,BERTCreationDataset,BERTEmbeddingsExtractor,BertCollateFunction,ContrastiveMSELoss,OptunaFunction,load,Test,RecipesEmbeddingsExtractor,ScalerInferenceUsers,ScalerInferenceRecipes,creation_recipes_set,CollateFunctionInferenceRecipes,CollateFunctionInferenceUsers
import optuna
from loguru import logger
import gc
import os
import plotly.io as pio
import joblib
from mlflow.tracking import MlflowClient
pio.renderers.default = "vscode"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
seed=42
set_seed(seed=seed)

In [20]:
mlflow.set_tracking_uri("file:./mlflow_runs/mlruns")
mlflow.set_experiment("Recipes_Recommendation_System_model_training")

c:\Users\sh032\anaconda3\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning:

The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.



<Experiment: artifact_location=('file:///C:/Users/sh032/OneDrive - HEC Montréal/Documents/Master 1/Recipe '
 'recommendation system/mlflow_runs/mlruns/503138512899622791'), creation_time=1775691667057, experiment_id='503138512899622791', last_update_time=1775691667057, lifecycle_stage='active', name='Recipes_Recommendation_System_model_training', tags={}, workspace='default'>

In [21]:
client=MlflowClient()

## Creation of the processed dataframe

In [3]:
interactions_train_df=pd.read_csv("data/interactions_train.csv")
PP_recipes_df=pd.read_csv("data/PP_recipes.csv")
PP_users_df=pd.read_csv("data/PP_users.csv")
RAW_interactions_df=pd.read_csv("data/RAW_interactions.csv")
RAW_recipes_df=pd.read_csv("data/RAW_recipes.csv")


In [4]:
interactions_train_df_filtered=interactions_train_df.sample(n=20000,replace=False,random_state=seed)

In [5]:
print(interactions_train_df_filtered.shape)


(20000, 6)


In [6]:
RAW_recipes_df.rename(columns={"id":"recipe_id"},inplace=True)
PP_recipes_df.rename(columns={"techniques":"techniques_recipes"},inplace=True)
PP_users_df.rename(columns={"techniques":"techniques_users"},inplace=True)

In [7]:
print("interactions_train_df_filtered columns:")
print(interactions_train_df_filtered.columns)
print("------------------------------------------------------------------")
print("PP_recipes_df columns:")
print(PP_recipes_df.columns)
print("-------------------------------------------------------------------")
print("PP_users_df columns:")
print(PP_users_df.columns)
print("----------------------------------------------------------------------")
print("RAW_interactions_df columns:")
print(RAW_interactions_df.columns)
print("------------------------------------------------------------------------")
print("RAW_recipes_df columns:")
print(RAW_recipes_df.columns)

interactions_train_df_filtered columns:
Index(['user_id', 'recipe_id', 'date', 'rating', 'u', 'i'], dtype='object')
------------------------------------------------------------------
PP_recipes_df columns:
Index(['id', 'i', 'name_tokens', 'ingredient_tokens', 'steps_tokens',
       'techniques_recipes', 'calorie_level', 'ingredient_ids'],
      dtype='object')
-------------------------------------------------------------------
PP_users_df columns:
Index(['u', 'techniques_users', 'items', 'n_items', 'ratings', 'n_ratings'], dtype='object')
----------------------------------------------------------------------
RAW_interactions_df columns:
Index(['user_id', 'recipe_id', 'date', 'rating', 'review'], dtype='object')
------------------------------------------------------------------------
RAW_recipes_df columns:
Index(['name', 'recipe_id', 'minutes', 'contributor_id', 'submitted', 'tags',
       'nutrition', 'n_steps', 'steps', 'description', 'ingredients',
       'n_ingredients'],
      dty

In [8]:
full_df=interactions_train_df_filtered.merge(PP_recipes_df,on="i",how="left",suffixes=(None,"_1")).merge(PP_users_df,on="u",how="left",suffixes=(None,"_2")).merge(RAW_interactions_df,on=["user_id","recipe_id"],how="left",suffixes=(None,"_3")).merge(RAW_recipes_df,on=["recipe_id"],how="left",suffixes=(None,"_3"))

In [9]:
full_df=full_df.drop(columns=["review",'rating_3',"id","ingredients","submitted","contributor_id","date_3","date","steps_tokens","ingredient_tokens","name_tokens","u"],axis=1)
full_df.to_parquet(path="./modules/dataframes/full_df.parquet")

In [10]:
full_parquet_file=pq.ParquetFile("./modules/dataframes/full_df.parquet")
full_df=load(full_parquet_file)

In [11]:
print(f"Columns of the full dataframe:")
print(full_df.columns)
print("----------------------------------------")
print(f"Shape of the full dataframe:")
print(full_df.shape)
print("----------------------------------------")
print(f"Columns of the filtered interaction dataframe:")
print(interactions_train_df_filtered.shape)
print("----------------------------------------")
print(f"Head of the full  dataframe:")
display(full_df.head())
print("----------------------------------------")
for col in full_df.columns:
    print(f"Unique values of {col} :")
    print(full_df[col].unique())
    print("----------------------------------------")
for col in full_df.columns:
    print(f"The type of {col} is")
    print(full_df[col].apply(type).unique())
    print("---------------------------------")

Columns of the full dataframe:
Index(['user_id', 'recipe_id', 'rating', 'i', 'techniques_recipes',
       'calorie_level', 'ingredient_ids', 'techniques_users', 'items',
       'n_items', 'ratings', 'n_ratings', 'name', 'minutes', 'tags',
       'nutrition', 'n_steps', 'steps', 'description', 'n_ingredients'],
      dtype='object')
----------------------------------------
Shape of the full dataframe:
(20000, 20)
----------------------------------------
Columns of the filtered interaction dataframe:
(20000, 6)
----------------------------------------
Head of the full  dataframe:


,user_id,recipe_id,rating,i,techniques_recipes,calorie_level,ingredient_ids,techniques_users,items,n_items,ratings,n_ratings,name,minutes,tags,nutrition,n_steps,steps,description,n_ingredients
0,231160,124810,5.0,170682,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ...",0,"[4574, 1910, 6906, 2499, 63, 332, 6270, 7470, ...","[3, 0, 0, 0, 1, 0, 0, 0, 0, 2, 1, 0, 0, 0, 0, ...","[8911, 20238, 117521, 79316, 154563, 34302, 17...",7,"[5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0]",7,huckleberry or blueberry coffee cake,75,"['time-to-make', 'course', 'main-ingredient', ...","[174.1, 4.0, 92.0, 8.0, 7.0, 3.0, 11.0]",11,['beat margarine and cream cheese at medium sp...,"cooking light published this in their book, fi...",10
1,142629,31342,5.0,120865,"[1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ...",1,"[1168, 1284, 8021, 2499, 330, 6270, 3217, 2318...","[6, 0, 0, 3, 1, 0, 0, 0, 0, 6, 0, 0, 0, 0, 0, ...","[122019, 150060, 117906, 82202, 119046, 141911...",10,"[4.0, 5.0, 5.0, 5.0, 4.0, 5.0, 3.0, 5.0, 4.0, ...",10,blender quiche or whatever you have in your ...,65,"['weeknight', 'time-to-make', 'course', 'main-...","[308.4, 37.0, 8.0, 20.0, 21.0, 41.0, 3.0]",10,"['preheat oven to 350f', 'generously grease a ...",the great part is you can use whatever leftove...,12
2,822358,441841,4.0,23142,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1,"[1648, 1706, 3347, 5298, 3044, 4623, 2148, 6270]","[1, 1, 0, 2, 3, 0, 0, 0, 1, 5, 0, 1, 0, 0, 0, ...","[29761, 87944, 72370, 139097, 39470, 37755, 23...",17,"[4.0, 5.0, 5.0, 5.0, 4.0, 4.0, 4.0, 5.0, 5.0, ...",17,sweet orange coleslaw,10,"['15-minutes-or-less', 'time-to-make', 'course...","[290.1, 21.0, 71.0, 14.0, 7.0, 8.0, 13.0]",6,"['chop the apples', 'in a small bowl mix the o...",this recipe came from one of my mom's cookbook...,8
3,655579,211110,4.0,156541,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0,"[7031, 5006, 339, 3203, 5975, 6270, 590]","[10, 0, 0, 5, 11, 0, 0, 0, 0, 7, 1, 3, 0, 0, 1...","[34492, 71279, 73805, 68721, 114795, 35164, 33...",28,"[2.0, 3.0, 4.0, 2.0, 5.0, 5.0, 5.0, 5.0, 5.0, ...",28,wilted greens with garlic and balsamic vinegar,15,"['15-minutes-or-less', 'time-to-make', 'course...","[77.5, 10.0, 5.0, 5.0, 2.0, 4.0, 1.0]",5,"['cut the greens into strips 1 inch wide', 'in...",this simple but delicious side dish can be mad...,7
4,35140,114469,5.0,30839,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ...",2,"[1240, 6270, 5006, 1257, 2683, 5168, 800, 2461...","[176, 4, 2, 55, 72, 0, 1, 7, 4, 141, 13, 15, 0...","[121722, 36691, 151293, 154016, 1205, 12802, 1...",364,"[5.0, 5.0, 0.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, ...",364,chicken in the limelight,75,"['time-to-make', 'course', 'main-ingredient', ...","[645.9, 69.0, 11.0, 22.0, 80.0, 54.0, 4.0]",8,"['preheat oven to 350', 'grate lime peel and s...",very tender chicken soaked with flavor,9


----------------------------------------
Unique values of user_id :
[231160 142629 822358 ... 241982 164867 137854]
----------------------------------------
Unique values of recipe_id :
[124810  31342 441841 ...  98278 274099 283283]
----------------------------------------
Unique values of rating :
[5. 4. 3. 2. 0. 1.]
----------------------------------------
Unique values of i :
[170682 120865  23142 ... 108495 128044  25867]
----------------------------------------
Unique values of techniques_recipes :
['[1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]'
 '[1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]'
 '[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [12]:
PP_users_df["len_items"]=PP_users_df["items"].str.count(r",")+1
PP_recipes_df["len_ingredients"]=PP_recipes_df["ingredient_ids"].str.count(r",")+1
max_len_ingredients=PP_recipes_df["len_ingredients"].max()
max_len_items=PP_users_df["len_items"].max()
PP_users_df.drop(columns=["len_items"],inplace=True)
PP_recipes_df.drop(columns=["len_ingredients"],inplace=True)

In [13]:
before_split_users_processor=BeforeSplitPreprocessingUsersFeatures(full_df,max_len_items=max_len_items)
full_df_processed=before_split_users_processor.preprocessing()

In [14]:
before_split_recipes_processor=BeforeSplitPreprocessingRecipesFeatures(full_df_processed,max_len_ingredients=max_len_ingredients)
full_df_processed=before_split_recipes_processor.preprocessing()

In [15]:
full_df_processed["rating_scaled"]=full_df_processed["rating"]/5
full_df_processed["ratings_scaled"]=full_df_processed["ratings"].apply(lambda x: list(map(lambda x: x/5,x)))

In [16]:
PP_recipes_df["ingredient_ids_transformed"]=PP_recipes_df["ingredient_ids"].apply(lambda x: list(map(lambda x: int(x)+1,ast.literal_eval(x))))
all_ingredient_ids=np.unique(list(itertools.chain.from_iterable(list(PP_recipes_df["ingredient_ids_transformed"]))))
ingredient_continuous_ids=pd.Series([i for i in range(1,len(all_ingredient_ids)+1)],index=list(np.sort(all_ingredient_ids)))

In [20]:
def mapping_list(x:list,mapping_serie:pd.Series):
    return [mapping_serie.loc[i] if i in mapping_serie.index else 0 for i in x]

In [21]:
full_df_processed["ingredient_ids_continuous"]=full_df_processed["ingredient_ids"].apply(lambda x : mapping_list(x,ingredient_continuous_ids))

In [22]:
full_df_processed["full_text"]=full_df_processed["name"]+" [SEP] "+full_df_processed["description"]+" [SEP] "+full_df_processed["steps"]+" [SEP] "+full_df_processed["tags"]

In [ ]:
#full_df_processed.to_parquet(path="./modules/dataframes/full_df_processed.parquet")

In [ ]:
#full_processed_parquet_file = pq.ParquetFile("./modules/dataframes/full_df_processed.parquet")
#full_df_processed=load(full_processed_parquet_file)

In [ ]:
#df_text=full_df_processed["full_text"]

In [ ]:
#distilbert_model=DistilBertModel.from_pretrained("distilbert-base-uncased")
#tokenizer=AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [ ]:
#bert_dataset=BERTCreationDataset(df_text)

In [ ]:
#generator=torch.Generator()
#generator.manual_seed(seed)
#bert_collate_function_object=BertCollateFunction(tokenizer=tokenizer)
#bert_dataloader=DataLoader(dataset=bert_dataset,batch_size=32,shuffle=False,collate_fn=bert_collate_function_object.collate_fn,worker_init_fn=seed_worker,generator=generator)

In [ ]:
#device = (torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda" if torch.cuda.is_available() else "cpu"))
#bert_extractor=BERTEmbeddingsExtractor(bert_model=distilbert_model,device=device)
#bert_extractor.get_bert_embeddings(dataloader=bert_dataloader,path="./modules/BERT_embeddings")

In [23]:
all_cls_embeddings=torch.load("./modules/BERT_embeddings/all_cls_embeddings.pt")
all_mean_embeddings=torch.load("./modules/BERT_embeddings/all_mean_embeddings.pt")
print(all_cls_embeddings.shape)
print(all_mean_embeddings.shape)

torch.Size([20000, 768])
torch.Size([20000, 768])


In [24]:
train_df,val_df,test_df,train_cls_embeddings,val_cls_embeddings,test_cls_embeddings,train_mean_embeddings,val_mean_embeddings,test_mean_embeddings=split_df(full_df_processed,cls_embeddings=all_cls_embeddings,mean_embeddings=all_mean_embeddings,random_state=seed)

In [25]:
after_split_preprocessor_users=AfterSplitPreprocessingUsersFeatures(train_df)
train_df_processed=after_split_preprocessor_users.preprocessing(train_df)
val_df_processed=after_split_preprocessor_users.preprocessing(val_df)
test_df_processed=after_split_preprocessor_users.preprocessing(test_df)

In [26]:
after_split_preprocessor_recipes=AfterSplitPreprocessingRecipesFeatures(train_df)
train_df_processed=after_split_preprocessor_recipes.preprocessing(train_df_processed)
val_df_processed=after_split_preprocessor_recipes.preprocessing(val_df_processed)
test_df_processed=after_split_preprocessor_recipes.preprocessing(test_df_processed)

In [27]:
scaler_users=StandardScaler()
scaler_recipes=StandardScaler()
scale_processor=Scaler(scaler_users,scaler_recipes)
train_df_processed,val_df_processed,test_df_processed=scale_processor.scale(train_df_processed,val_df_processed,test_df_processed)

In [28]:
joblib.dump(scale_processor.scaler_users, "./modules/scalers/scaler_users.pkl")
joblib.dump(scale_processor.scaler_recipes, "./modules/scalers/scaler_recipes.pkl")

['./modules/scalers/scaler_recipes.pkl']

In [29]:
scaler_users=joblib.load("./modules/scalers/scaler_users.pkl")
scaler_recipes=joblib.load("./modules/scalers/scaler_recipes.pkl")

In [30]:
train_df.to_parquet(path="./modules/dataframes/train_df.parquet")

train_df_processed.to_parquet(path="./modules/dataframes/train_df_processed.parquet")
val_df_processed.to_parquet(path="./modules/dataframes/val_df_processed.parquet")
test_df_processed.to_parquet(path="./modules/dataframes/test_df_processed.parquet")

torch.save(train_cls_embeddings,f"./modules/BERT_embeddings/train_cls_embeddings.pt")
torch.save(val_cls_embeddings,f"./modules/BERT_embeddings/val_cls_embeddings.pt")
torch.save(test_cls_embeddings,f"./modules/BERT_embeddings/test_cls_embeddings.pt")

torch.save(train_mean_embeddings,f"./modules/BERT_embeddings/train_mean_embeddings.pt")
torch.save(val_mean_embeddings,f"./modules/BERT_embeddings/val_mean_embeddings.pt")
torch.save(test_mean_embeddings,f"./modules/BERT_embeddings/test_mean_embeddings.pt")


## Load of the dataframes

In [34]:
full_parquet_file=pq.ParquetFile("./modules/dataframes/full_df.parquet")
full_processed_parquet_file=pq.ParquetFile("./modules/dataframes/full_df_processed.parquet")
train_parquet_file=pq.ParquetFile("./modules/dataframes/train_df.parquet")
train_processed_parquet_file = pq.ParquetFile("./modules/dataframes/train_df_processed.parquet")
val_processed_parquet_file= pq.ParquetFile("./modules/dataframes/val_df_processed.parquet")
test_processed_parquet_file= pq.ParquetFile("./modules/dataframes/test_df_processed.parquet")

full_df=load(full_parquet_file)
full_df_processed=load(full_processed_parquet_file)
train_df=load(train_parquet_file)
train_df_processed =load(train_processed_parquet_file)
val_df_processed=load(val_processed_parquet_file)
test_df_processed=load(test_processed_parquet_file)

train_cls_embeddings=torch.load(f"./modules/BERT_embeddings/train_cls_embeddings.pt")
val_cls_embeddings=torch.load(f"./modules/BERT_embeddings/val_cls_embeddings.pt")
test_cls_embeddings=torch.load(f"./modules/BERT_embeddings/test_cls_embeddings.pt")

train_mean_embeddings=torch.load(f"./modules/BERT_embeddings/train_mean_embeddings.pt")
val_mean_embeddings=torch.load(f"./modules/BERT_embeddings/val_mean_embeddings.pt")
test_mean_embeddings=torch.load(f"./modules/BERT_embeddings/test_mean_embeddings.pt")

all_cls_embeddings=torch.load("./modules/BERT_embeddings/all_cls_embeddings.pt")
all_mean_embeddings=torch.load("./modules/BERT_embeddings/all_mean_embeddings.pt")

In [39]:
print(f"Columns of the full processed dataframe:")
print(train_df_processed.columns)
print("----------------------------------------")
print(f"Shape of the full processed dataframe:")
print(train_df_processed.shape)
print("----------------------------------------")
print(f"Head of the full  processed dataframe:")
display(train_df_processed.head())
print("----------------------------------------")
for col in train_df_processed.columns:
    print(f"Exemple of {col} :")
    print(train_df_processed.iloc[0][col])
    print("----------------------------------------")
for col in train_df_processed.columns:
    print(f"The type of {col} is")
    print(train_df_processed[col].apply(type).unique())
    print("---------------------------------")

Columns of the full processed dataframe:
Index(['user_id', 'recipe_id', 'rating', 'i', 'techniques_recipes',
       'calorie_level', 'ingredient_ids', 'techniques_users', 'items',
       'n_items', 'ratings', 'n_ratings', 'name', 'minutes', 'tags',
       'nutrition', 'n_steps', 'steps', 'description', 'n_ingredients',
       'rating_scaled', 'ratings_scaled', 'ingredient_ids_continuous',
       'full_text', 'n_items_scaled', 'n_ratings_scaled',
       'calorie_level_scaled', 'minutes_scaled', 'n_steps_scaled',
       'n_ingredients_scaled'],
      dtype='object')
----------------------------------------
Shape of the full processed dataframe:
(14000, 30)
----------------------------------------
Head of the full  processed dataframe:


,user_id,recipe_id,rating,i,techniques_recipes,calorie_level,ingredient_ids,techniques_users,items,n_items,...,rating_scaled,ratings_scaled,ingredient_ids_continuous,full_text,n_items_scaled,n_ratings_scaled,calorie_level_scaled,minutes_scaled,n_steps_scaled,n_ingredients_scaled
0,175728,11770,5,109126,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ...",0,"[4624, 6655, 217, 6927, 6336, 4864, 0, 0, 0, 0...","[87, 2, 0, 27, 42, 1, 1, 7, 0, 80, 3, 3, 1, 0,...","[154350, 54820, 77571, 45268, 124558, 38966, 1...",202,...,1.0,"[1.0, 1.0, 0.8, 1.0, 0.8, 1.0, 0.8, 0.6, 0.8, ...","[4609, 6629, 215, 6900, 6313, 4847, 0, 0, 0, 0...",walnut brewery asiago cheese dip [SEP] this a ...,-0.386555,-0.386555,-1.086207,-0.156327,0.050739,-0.923525
1,50510,350360,4,53777,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",2,"[5314, 5007, 3204, 5011, 7214, 589, 2595, 2857...","[127, 1, 2, 91, 191, 0, 0, 14, 3, 176, 10, 12,...","[59108, 117303, 16669, 165155, 78915, 115100, ...",580,...,0.8,"[0.8, 1.0, 1.0, 0.8, 1.0, 0.8, 0.8, 1.0, 1.0, ...","[5295, 4990, 3195, 4994, 7187, 586, 2588, 2849...",nif s penne with feta [SEP] a delicious and fr...,0.004451,0.004451,1.461012,-0.142037,-0.715266,0.643517
2,9749,13099,4,13490,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1,"[151, 1125, 5299, 1910, 4764, 156, 6907, 6277,...","[44, 1, 1, 23, 33, 0, 1, 5, 2, 35, 10, 5, 0, 0...","[75852, 104812, 129769, 99788, 101820, 166815,...",144,...,0.8,"[0.8, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.8, 1.0, ...","[149, 1119, 5281, 1903, 4748, 154, 6880, 6254,...",mom s waldorf salad [SEP] this is my mother's ...,-0.446551,-0.446551,0.187403,-0.184908,-0.970601,-0.296708
3,161283,56695,0,115884,"[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1,"[2731, 3485, 2405, 2500, 6271, 3487, 64, 4054,...","[33, 1, 0, 10, 12, 0, 0, 5, 0, 30, 6, 3, 0, 0,...","[130597, 16122, 49703, 152649, 103416, 36541, ...",92,...,0.0,"[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[2723, 3476, 2398, 2493, 6248, 3478, 63, 4041,...",the very best salisbury steak [SEP] this is ou...,-0.500340,-0.500340,0.187403,-0.099165,-0.204596,0.643517
4,463436,286054,4,60799,"[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0,"[4718, 7999, 3045, 3830, 0, 0, 0, 0, 0, 0, 0, ...","[111, 1, 2, 69, 82, 0, 1, 9, 1, 114, 17, 20, 1...","[32365, 145711, 157654, 141020, 35883, 67138, ...",395,...,0.8,"[1.0, 1.0, 1.0, 1.0, 1.0, 0.8, 1.0, 1.0, 1.0, ...","[4702, 7969, 3036, 3820, 0, 0, 0, 0, 0, 0, 0, ...",o j yogurt shake [SEP] this is a yummy and ...,-0.186914,-0.186914,-1.086207,-0.227780,-0.970601,-1.550341


----------------------------------------
Exemple of user_id :
175728
----------------------------------------
Exemple of recipe_id :
11770
----------------------------------------
Exemple of rating :
5
----------------------------------------
Exemple of i :
109126
----------------------------------------
Exemple of techniques_recipes :
[1 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0]
----------------------------------------
Exemple of calorie_level :
0
----------------------------------------
Exemple of ingredient_ids :
[4624 6655  217 6927 6336 4864    0    0    0    0    0    0    0    0
    0    0    0    0    0    0]
----------------------------------------
Exemple of techniques_users :
[87  2  0 27 42  1  1  7  0 80  3  3  1  0  3  0 40  0  0  5 20  8  2  8
  9  0  4  4 43  4  1  1  1 58  0  3 19  6 14  1  1  6 29 35  5  1 20  3
  0  5  2  3  0 18  5 18 10 20]
----------------------------------------
Exemple of 

## Model

In [40]:
train_dataset=CreationDataset(train_df_processed,cls_embeddings=train_cls_embeddings,mean_embeddings=train_mean_embeddings)
val_dataset=CreationDataset(val_df_processed,cls_embeddings=val_cls_embeddings,mean_embeddings=val_mean_embeddings)

In [41]:
train_dataset.__getitem__(10)

{'user_id': 440300,
 'recipe_id': 5013,
 'rating': 5,
 'i': 109198,
 'techniques_recipes': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1], dtype=int64),
 'calorie_level': 1,
 'ingredient_ids': array([  64, 5007, 2500, 7169, 7656, 7957, 6907,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0]),
 'techniques_users': array([5, 0, 0, 0, 2, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1,
        0, 0, 1, 0, 0, 1, 2, 0, 0, 0, 0, 2, 0, 0, 1, 0, 1, 0, 0, 1, 1, 2,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 2], dtype=int64),
 'items': array([ 18676,  75351, 109198, ...,      0,      0,      0]),
 'n_items': 10,
 'ratings': array([2, 5, 5, ..., 0, 0, 0]),
 'n_ratings': 10,
 'name': 'chicago style deep dish pizza crust',
 'minutes': 0,
 'tags': '15 minutes or less [SEP] time to make [SEP] course [SEP] cuisine

In [42]:
generator=torch.Generator()
generator.manual_seed(seed)
collate_function_object=CollateFunction()
collate_fn=collate_function_object.collate_fn
train_dataloader=DataLoader(train_dataset,batch_size=64,collate_fn=collate_fn,generator=generator,worker_init_fn=seed_worker,shuffle=True,drop_last=True)
val_dataloader=DataLoader(val_dataset,batch_size=64,collate_fn=collate_fn,generator=generator,worker_init_fn=seed_worker,shuffle=True,drop_last=True)

In [43]:
batch=next(iter(train_dataloader))
for key in list(batch.keys()):
    print(key)
    print(f"The shape of batch {key} is :")
    print(batch[key].shape)
    print("-------------------")

user_id
The shape of batch user_id is :
torch.Size([64, 1])
-------------------
recipe_id
The shape of batch recipe_id is :
torch.Size([64, 1])
-------------------
rating_scaled
The shape of batch rating_scaled is :
torch.Size([64, 1])
-------------------
i
The shape of batch i is :
torch.Size([64, 1])
-------------------
technique_recipes
The shape of batch technique_recipes is :
torch.Size([64, 58])
-------------------
calorie_level_scaled
The shape of batch calorie_level_scaled is :
torch.Size([64, 1])
-------------------
ingredient_ids
The shape of batch ingredient_ids is :
torch.Size([64, 20])
-------------------
techniques_users
The shape of batch techniques_users is :
torch.Size([64, 58])
-------------------
items
The shape of batch items is :
torch.Size([64, 6437])
-------------------
n_items_scaled
The shape of batch n_items_scaled is :
torch.Size([64, 1])
-------------------
ratings_scaled
The shape of batch ratings_scaled is :
torch.Size([64, 6437])
-------------------
n_rat

In [ ]:
#PP_recipes_df["ingredient_ids_transformed"]=PP_recipes_df["ingredient_ids"].apply(lambda x: list(map(int,ast.literal_eval(x))))
#all_ingredient_ids=np.unique(list(itertools.chain.from_iterable(list(PP_recipes_df["ingredient_ids_transformed"]))))
#ingredient_continuous_ids=pd.Series([i for i in range(1,len(all_ingredient_ids)+1)],index=list(np.sort(all_ingredient_ids)))
#ingredient_continuous_ids.to_pickle("./modules/dataframes/ingredient_continuous_ids.pkl")

In [ ]:
#ingredient_continuous_ids=pd.read_pickle("./modules/dataframes/ingredient_continuous_ids.pkl")
#n_recipes_ids=len(np.unique(PP_recipes_df["i"]))
#n_ingredients_ids=len(all_ingredient_ids)

#print(f"There are {n_recipes_ids} recipe ids")
#print(f"There are {n_ingredients_ids} ingredient ids")

There are 178265 recipe ids
There are 7993 ingredient ids


In [ ]:
#hash_encoder=EncodedHashedEmbeddings(n_recipes_ids,512,n_ingredients_ids,512)
#hashed_recipes_ids_encoded_embeddings,hashed_ingredients_ids_encoded_embeddings=hash_encoder.get_encoded_hashed_embeddings("./modules/hashed_encoded_tables")

In [48]:
hashed_ingredients_ids_encoded_embeddings=torch.load("./modules/hashed_encoded_tables/hashed_embeddings_ingredients.pt")
hashed_recipes_ids_encoded_embeddings=torch.load("./modules/hashed_encoded_tables/hashed_embeddings_recipes.pt")

In [50]:
print(hashed_recipes_ids_encoded_embeddings.shape)
print(hashed_ingredients_ids_encoded_embeddings.shape)

torch.Size([178267, 512])
torch.Size([7995, 512])


In [ ]:
#Loss with alpha = 0.3 and temperature = 0.2
loss_alpha=0.3
temperature=0.2
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
hyperparemeters_ranges={
    "batch_size":[32, 64],
    "dropout":{
        "low":0.05,
        "high":0.3,
        "step":0.05
    },
    "projec_dropout":{
        "low":0.05,
        "high":0.3,
        "step":0.05
    },
    "lr":{
        "low":1e-4,
        "high":1e-2,
        "log":True
    },
    "weight_decay":{
        "low":1e-5,
        "high":1e-3,
        "log":True
    },
    "warmup_prop":{
        "low":0.01,
        "high":0.1,
        "step":0.01
    },
    "mean_mode": [True, False]
}
optuna_object=OptunaFunction(train_df=train_df_processed,val_df=val_df_processed,train_cls_embeddings=train_cls_embeddings,
                             val_cls_embeddings=val_cls_embeddings,train_mean_embeddings=train_mean_embeddings,
                             val_mean_embeddings=val_mean_embeddings,hashed_ingredients_ids_encoded_embeddings=hashed_ingredients_ids_encoded_embeddings,
                             hashed_recipes_ids_encoded_embeddings=hashed_recipes_ids_encoded_embeddings,loss_alpha=loss_alpha,temperature=temperature,
                             device=device,hyperparemeters_ranges=hyperparemeters_ranges)

In [30]:
study=optuna.create_study(direction="minimize",pruner=optuna.pruners.MedianPruner(),storage="sqlite:///optuna_study.db",load_if_exists=True,study_name="recommendation_model_study")

[I 2026-04-08 23:50:21,903] Using an existing study with name 'recommendation_model_study' instead of creating a new one.


In [31]:
study.optimize(optuna_object.objective,n_trials=10)

2026-04-08 23:50:40.895 | INFO     | modules.optuna_function:objective:88 - Trial number 0: 
2026-04-08 23:50:40.896 | INFO     | modules.optuna_function:objective:89 - batch_size:64_dropout:0,15000000000000002_projec_dropout:0,1_lr:0,0004724968056532259_weight_decay:1,371295256336384e-05_warmup_prop:0,060000000000000005_mean_mode:False_loss_temp:0,2_loss_alpha:0,3
2026-04-08 23:50:40.897 | INFO     | modules.train:run_training:89 - Epoch 0 : 
2026-04-08 23:51:49.959 | INFO     | modules.train:run_training:180 - Epoch 0 : train total loss = 1.178026832571817
2026-04-08 23:51:49.960 | INFO     | modules.train:run_training:181 - Epoch 0 : train contrastive loss = 3.817592550855164
2026-04-08 23:51:49.961 | INFO     | modules.train:run_training:182 - Epoch 0 : train MSE loss = 0.04678431115702752
2026-04-08 23:51:49.962 | INFO     | modules.train:run_training:184 - Epoch 0 : validation total loss = 1.163207067095715
2026-04-08 23:51:49.962 | INFO     | modules.train:run_training:185 - Epo

----------------------------------------------------------------------------------------------------


[I 2026-04-08 23:58:00,644] Trial 0 finished with value: 1.1344977643178857 and parameters: {'batch_size': 64, 'dropout': 0.15000000000000002, 'projec_dropout': 0.1, 'lr': 0.0004724968056532259, 'weight_decay': 1.371295256336384e-05, 'warmup_prop': 0.060000000000000005, 'mean_mode': False}. Best is trial 0 with value: 1.1344977643178857.
2026-04-08 23:58:01.305 | INFO     | modules.optuna_function:objective:88 - Trial number 1: 
2026-04-08 23:58:01.306 | INFO     | modules.optuna_function:objective:89 - batch_size:64_dropout:0,05_projec_dropout:0,2_lr:0,00013340658743746087_weight_decay:3,6677632134421984e-05_warmup_prop:0,08_mean_mode:True_loss_temp:0,2_loss_alpha:0,3
2026-04-08 23:58:01.307 | INFO     | modules.train:run_training:89 - Epoch 0 : 
2026-04-08 23:59:13.465 | INFO     | modules.train:run_training:180 - Epoch 0 : train total loss = 1.1853736562466404
2026-04-08 23:59:13.467 | INFO     | modules.train:run_training:181 - Epoch 0 : train contrastive loss = 3.8196088633406053


----------------------------------------------------------------------------------------------------


[I 2026-04-09 00:06:35,156] Trial 1 finished with value: 1.1371555121048638 and parameters: {'batch_size': 64, 'dropout': 0.05, 'projec_dropout': 0.2, 'lr': 0.00013340658743746087, 'weight_decay': 3.6677632134421984e-05, 'warmup_prop': 0.08, 'mean_mode': True}. Best is trial 0 with value: 1.1344977643178857.
2026-04-09 00:06:35.733 | INFO     | modules.optuna_function:objective:88 - Trial number 2: 
2026-04-09 00:06:35.734 | INFO     | modules.optuna_function:objective:89 - batch_size:32_dropout:0,1_projec_dropout:0,2_lr:0,0005119178802555773_weight_decay:1,3386144108743617e-05_warmup_prop:0,09999999999999999_mean_mode:True_loss_temp:0,2_loss_alpha:0,3
2026-04-09 00:06:35.735 | INFO     | modules.train:run_training:89 - Epoch 0 : 
2026-04-09 00:07:46.680 | INFO     | modules.train:run_training:180 - Epoch 0 : train total loss = 0.9828978645992498
2026-04-09 00:07:46.681 | INFO     | modules.train:run_training:181 - Epoch 0 : train contrastive loss = 3.171550656347035
2026-04-09 00:07:4

...

[I 2026-04-09 00:34:11,392] Trial 5 finished with value: 0.9498907532743228 and parameters: {'batch_size': 32, 'dropout': 0.05, 'projec_dropout': 0.15000000000000002, 'lr': 0.0010036326863450127, 'weight_decay': 0.00013814862105675894, 'warmup_prop': 0.08, 'mean_mode': False}. Best is trial 2 with value: 0.9452530517373033.
2026-04-09 00:34:12.007 | INFO     | modules.optuna_function:objective:88 - Trial number 6: 
2026-04-09 00:34:12.008 | INFO     | modules.optuna_function:objective:89 - batch_size:32_dropout:0,1_projec_dropout:0,1_lr:0,0017263266493212528_weight_decay:0,000683227867921707_warmup_prop:0,03_mean_mode:True_loss_temp:0,2_loss_alpha:0,3
2026-04-09 00:34:12.010 | INFO     | modules.train:run_training:89 - Epoch 0 : 
2026-04-09 00:35:21.755 | INFO     | modules.train:run_training:180 - Epoch 0 : train total loss = 0.9749773392961009
2026-04-09 00:35:21.756 | INFO     | modules.train:run_training:181 - Epoch 0 : train contrastive loss = 3.1586894907176633
2026-04-09 00:35:2

----------------------------------------------------------------------------------------------------


[I 2026-04-09 00:40:14,198] Trial 6 finished with value: 0.9534697628790333 and parameters: {'batch_size': 32, 'dropout': 0.1, 'projec_dropout': 0.1, 'lr': 0.0017263266493212528, 'weight_decay': 0.000683227867921707, 'warmup_prop': 0.03, 'mean_mode': True}. Best is trial 2 with value: 0.9452530517373033.
2026-04-09 00:40:14.801 | INFO     | modules.optuna_function:objective:88 - Trial number 7: 
2026-04-09 00:40:14.802 | INFO     | modules.optuna_function:objective:89 - batch_size:32_dropout:0,1_projec_dropout:0,1_lr:0,001532223306220127_weight_decay:0,0001182983291048804_warmup_prop:0,01_mean_mode:True_loss_temp:0,2_loss_alpha:0,3
2026-04-09 00:40:14.803 | INFO     | modules.train:run_training:89 - Epoch 0 : 
2026-04-09 00:41:25.783 | INFO     | modules.train:run_training:180 - Epoch 0 : train total loss = 0.9795371995092257
2026-04-09 00:41:25.784 | INFO     | modules.train:run_training:181 - Epoch 0 : train contrastive loss = 3.177291827016346
2026-04-09 00:41:25.785 | INFO     | mo

----------------------------------------------------------------------------------------------------


[I 2026-04-09 00:48:43,666] Trial 9 finished with value: 0.9553154892818903 and parameters: {'batch_size': 32, 'dropout': 0.1, 'projec_dropout': 0.25, 'lr': 0.006987254241678245, 'weight_decay': 1.2335097316744706e-05, 'warmup_prop': 0.09999999999999999, 'mean_mode': True}. Best is trial 2 with value: 0.9452530517373033.


In [4]:
study_loaded=optuna.load_study(study_name="recommendation_model_study",storage="sqlite:///optuna_study.db")

In [ ]:
"""
study_best_components={
    "study_best_params":study_loaded.best_params,
    "study_best_value":study_loaded.best_value
}
torch.save(study_best_components,"./train_savings/study_best_components/study_best_components.pt")
"""

In [5]:
fig=optuna.visualization.plot_param_importances(study_loaded)
fig.show()

In [6]:
fig=optuna.visualization.plot_parallel_coordinate(study_loaded)
fig.show()

In [7]:
fig=optuna.visualization.plot_optimization_history(study_loaded)
fig.show()

In [ ]:
#best_model_name="Best_Recommendation_System_model"
#best_model_uri=f"runs:/269606da7f6c42c8a2d9ba6ae0bd2025/model_2"
#registration=mlflow.register_model(best_model_uri,best_model_name)

Registered model 'Best_Recommendation_System_model' already exists. Creating a new version of this model...
2026/04/09 20:14:34 WARNING mlflow.tracking._model_registry.fluent: Run with id 269606da7f6c42c8a2d9ba6ae0bd2025 has no artifacts at artifact path 'model_2', registering model based on models:/m-d7b579ae6cd244c2b24af301bf99847e instead
Created version '1' of model 'Best_Recommendation_System_model'.


In [ ]:
"""
client.transition_model_version_stage(
    name="Best_Recommendation_System_model",
    version=1,
    stage="Staging"
)
"""

C:\Users\sh032\AppData\Local\Temp\ipykernel_19112\3661356175.py:1: FutureWarning:

``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages



<ModelVersion: aliases=[], creation_timestamp=1775780074938, current_stage='Staging', deployment_job_state=None, description=None, last_updated_timestamp=1775780528756, metrics=[<Metric: dataset_digest=None, dataset_name=None, key='best_validation_contrastive_loss', model_id='m-d7b579ae6cd244c2b24af301bf99847e', run_id='269606da7f6c42c8a2d9ba6ae0bd2025', step=0, timestamp=1775693610145, value=3.055713999655939>,
 <Metric: dataset_digest=None, dataset_name=None, key='best_validation_MSE_loss', model_id='m-d7b579ae6cd244c2b24af301bf99847e', run_id='269606da7f6c42c8a2d9ba6ae0bd2025', step=0, timestamp=1775693610145, value=0.04076973832542858>,
 <Metric: dataset_digest=None, dataset_name=None, key='best_validation_total_loss', model_id='m-d7b579ae6cd244c2b24af301bf99847e', run_id='269606da7f6c42c8a2d9ba6ae0bd2025', step=0, timestamp=1775693610145, value=0.9452530517373033>,
 <Metric: dataset_digest=None, dataset_name=None, key='epoch_train_contrastive_loss', model_id='m-d7b579ae6cd244c2b24

In [ ]:
#model_version=client.get_model_version("Best_Recommendation_System_model",1)
#version=int(model_version.version)
#client.set_model_version_tag(name="Best_Recommendation_System_model",key="role",value="Challenger",version=version)
#client.set_model_version_tag(name="Best_Recommendation_System_model",key="Stage",value="Staging",version=version)
#client.set_registered_model_alias(name="Best_Recommendation_System_model",alias="challenger",version=1)

In [ ]:
device = (torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda" if torch.cuda.is_available() else "cpu"))

model_version=client.get_model_version("Best_Recommendation_System_model",1)
best_model_registered_uri=f"models:/Best_Recommendation_System_model/1"
best_recommendation_model=mlflow.pytorch.load_model(best_model_registered_uri,map_location=torch.device("cpu"))
best_recommendation_model.device=device


batch_size=int(model_version.params["batch_size"])
loss_temperature=float(model_version.params["loss_temperature"])
loss_alpha=float(model_version.params["loss_alpha"])
run_id=model_version.run_id

tokenizer=AutoTokenizer.from_pretrained("distilbert-base-uncased")

test_dataset=CreationDataset(test_df_processed,test_cls_embeddings,test_mean_embeddings)
generator=torch.Generator()
generator.manual_seed(seed)
collate_function_object=CollateFunction()
collate_fn=collate_function_object.collate_fn
test_dataloader=DataLoader(test_dataset,batch_size=batch_size,collate_fn=collate_fn,generator=generator,worker_init_fn=seed_worker,shuffle=True,drop_last=True)

tester=Test(test_dataloader=test_dataloader,model=best_recommendation_model,device=device,loss_alpha=loss_alpha,loss_temperature=loss_temperature)
test_total_loss,test_contrastive_loss,test_mse_loss=tester.run_testing(f"./test_savings",run_id=run_id)

c:\Users\sh032\anaconda3\Lib\site-packages\mlflow\tracking\_model_registry\utils.py:220: FutureWarning:

The filesystem model registry backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.

2026/04/12 20:12:52 WARNING mlflow.pytorch: Stored model version '2.10.0+cu128' does not match installed PyTorch version '2.8.0+cpu'
2026-04-12 20:12:52.876 | INFO     | modules.test:run_testing:25 - Start of the test :
2026-04-12 20:15:08.103 | INFO     | modules.test:run_testing:62 - Test Total loss = 0.9561376308882108
2026-04-12 20:15:08.106 | INFO     | modules.test:run_testing:63 - Test Contrastive loss = 3.08149286495742
2026-04-12 20:15:08.106 | INFO     | modules.test:run_testing:64 - Test MSE loss = 0.045271052628435116


In [ ]:
"""
model_version=client.get_model_version("Best_Recommendation_System_model",1)
version=int(model_version.version)
test_metrics=torch.load("./test_savings/test_metrics.pt",weights_only=False)
for key,value in test_metrics.items():
    client.set_model_version_tag("Best_Recommendation_System_model",version=1,key=key,value=value)
"""

In [5]:
recipes_df,recipes_set_cls_embeddings,recipes_set_mean_embeddings=creation_recipes_set(full_df,all_cls_embeddings,all_mean_embeddings,path="./modules/recipes_set")

In [6]:
print(recipes_df.shape)
print(recipes_set_cls_embeddings.shape)
print(recipes_set_mean_embeddings.shape)

(15423, 13)
torch.Size([15423, 768])
torch.Size([15423, 768])


In [7]:
PP_recipes_df=pd.read_csv("data/PP_recipes.csv")
PP_recipes_df["len_ingredients"]=PP_recipes_df["ingredient_ids"].str.count(r",")+1
max_len_ingredients=PP_recipes_df["len_ingredients"].max()
PP_recipes_df.drop(columns=["len_ingredients"],inplace=True)

In [8]:
recipe_before_split_processor=BeforeSplitPreprocessingRecipesFeatures(recipes_df,max_len_ingredients)
recipes_df_processed=recipe_before_split_processor.preprocessing()

In [ ]:
PP_recipes_df["ingredient_ids_transformed"]=PP_recipes_df["ingredient_ids"].apply(lambda x: list(map(lambda x: int(x)+1,ast.literal_eval(x))))
all_ingredient_ids=np.unique(list(itertools.chain.from_iterable(list(PP_recipes_df["ingredient_ids_transformed"]))))
ingredient_continuous_ids=pd.Series([i for i in range(1,len(all_ingredient_ids)+1)],index=list(np.sort(all_ingredient_ids)))

In [ ]:
def mapping_list(x:list,mapping_serie:pd.Series):
    return [mapping_serie.loc[i] if i in mapping_serie.index else 0 for i in x]
recipes_df_processed["ingredient_ids_continuous"]=recipes_df_processed["ingredient_ids"].apply(lambda x : mapping_list(x,ingredient_continuous_ids))

In [ ]:
recipes_df_processed["full_text"]=recipes_df_processed["name"]+" [SEP] "+recipes_df_processed["description"]+" [SEP] "+recipes_df_processed["steps"]+" [SEP] "+recipes_df_processed["tags"]

In [10]:
recipe_after_split_processor=AfterSplitPreprocessingRecipesFeatures(train_df)
recipes_df_processed=recipe_after_split_processor.preprocessing(recipes_df_processed)

In [11]:
scaler_recipes=joblib.load("./modules/scalers/scaler_recipes.pkl")
recipes_scaler=ScalerInferenceRecipes(scaler_recipes)
recipes_df_processed=recipes_scaler.scale(recipes_df_processed)

In [12]:
recipes_df_processed.to_parquet("./modules/recipes_set/recipes_df_processed.parquet")

In [4]:
recipes_processed_parquet_file=pq.ParquetFile("./modules/recipes_set/recipes_df_processed.parquet")
recipes_parquet_file=pq.ParquetFile("./modules/recipes_set/recipes_df.parquet")

recipes_df=load(recipes_parquet_file)
recipes_df_processed=load(recipes_processed_parquet_file)
recipes_set_cls_embeddings=torch.load("./modules/recipes_set/recipes_set_cls_embeddings.pt")
recipes_set_mean_embeddings=torch.load("./modules/recipes_set/recipes_set_mean_embeddings.pt")

In [5]:
print(recipes_df.columns)

Index(['recipe_id', 'i', 'name', 'description', 'tags', 'calorie_level',
       'minutes', 'n_ingredients', 'ingredient_ids', 'nutrition', 'n_steps',
       'steps', 'techniques_recipes'],
      dtype='object')


In [14]:
print(recipes_df_processed.columns)
print(recipes_df_processed.shape)
print(recipes_set_cls_embeddings.shape)
print(recipes_set_mean_embeddings.shape)


Index(['recipe_id', 'i', 'name', 'description', 'tags', 'calorie_level',
       'minutes', 'n_ingredients', 'ingredient_ids', 'nutrition', 'n_steps',
       'steps', 'techniques_recipes', 'ingredient_ids_continuous', 'full_text',
       'calorie_level_scaled', 'minutes_scaled', 'n_steps_scaled',
       'n_ingredients_scaled'],
      dtype='object')
(15423, 19)
torch.Size([15423, 768])
torch.Size([15423, 768])


In [27]:
device = (torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda" if torch.cuda.is_available() else "cpu"))

model_version=client.get_model_version("Best_Recommendation_System_model",1)
best_model_registered_uri=f"models:/Best_Recommendation_System_model/1"
best_recommendation_model=mlflow.pytorch.load_model(best_model_registered_uri,map_location=torch.device("cpu"))
best_recommendation_model.device=device

batch_size=int(model_version.params["batch_size"])
recipes_dataset=CreationDataset(recipes_df_processed,recipes_set_cls_embeddings,recipes_set_mean_embeddings)
generator=torch.Generator()
generator.manual_seed(seed)
collate_function_object=CollateFunctionInferenceRecipes()
collate_fn=collate_function_object.collate_fn
recipes_dataloader=DataLoader(recipes_dataset,batch_size=batch_size,collate_fn=collate_fn,generator=generator,worker_init_fn=seed_worker,shuffle=False,drop_last=False)


recipes_embeddings_extractor=RecipesEmbeddingsExtractor(model=best_recommendation_model,device=device)
recipes_embeddings=recipes_embeddings_extractor.get_recipes_embeddings(dataloader=recipes_dataloader,path=f"./modules/recipes_embeddings")

2026/04/15 22:09:27 WARNING mlflow.pytorch: Stored model version '2.10.0+cu128' does not match installed PyTorch version '2.8.0+cpu'


In [28]:
print(recipes_embeddings.shape)

torch.Size([15423, 256])


In [29]:
recipes_embeddings=torch.load(f"./modules/recipes_embeddings/recipes_embeddings.pt")

In [3]:
distilbert_model=DistilBertModel.from_pretrained("distilbert-base-uncased")
tokenizer=AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [4]:
full_text="abab ansjkan d abia"
tokenized_full_text=tokenizer(full_text,return_tensors="pt",max_length=512,truncation=True,padding="max_length")

In [12]:
output=distilbert_model(input_ids=tokenized_full_text["input_ids"],attention_mask=tokenized_full_text["attention_mask"]).last_hidden_state

In [14]:
print(output.shape)

torch.Size([1, 512, 768])
